# NB_lab_test_trends
Builds **`<catalog>.gold.lab_test_trends`** from the silver layer, as defined in the `sttm_silver_to_gold.xlsx` STTM (sheet `lab_test_trends`).

* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `silver.lab_results` grouped by `test_name`, year/month of `test_date`
* **Joins:** none required; optional LEFT JOIN `silver.calendar` on the month-start date key (for `month_name` / `quarter_name`)
* **Window function:** `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year`, `month` (month-over-month trend)
* **Filters:** none
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

Note: the STTM names the calendar key `date_key`; the actual column in `silver.calendar` is `datekey` (INT, yyyyMMdd). The join uses `datekey` of the first day of the month (`make_date(year, month, 1)`), which is equivalent to `date_trunc('month', test_date)`.

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"

print("=" * 80)
print("STEP 1: Input parameters")
print("=" * 80)
print(f"  catalog       : '{catalog}'")
print(f"  Source schema : {source_schema}")
print(f"  Target table  : {target_table}")

In [0]:
lab_test_trends_sql = f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH monthly AS (
  -- Lab results aggregated to test_name x month grain
  SELECT
    test_name,
    YEAR(test_date)                       AS year,
    MONTH(test_date)                      AS month,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM {source_schema}.lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  -- Month-over-month trend: previous month's abnormal rate for the same test
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  r.test_name,
  CAST(r.year AS INT)                                                        AS year,
  CAST(r.month AS INT)                                                       AS month,
  c.month_name,
  c.quarter_name,
  CAST(r.total_tests AS INT)                                                 AS total_tests,
  CAST(r.abnormal_test_count AS INT)                                         AS abnormal_test_count,
  CAST(r.critical_test_count AS INT)                                         AS critical_test_count,
  r.abnormal_rate_pct,
  r.critical_rate_pct,
  CAST(r.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                       AS prev_month_abnormal_rate_pct,
  CAST(r.abnormal_rate_pct - r.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag r
LEFT JOIN {source_schema}.calendar c
  ON c.datekey = CAST(date_format(make_date(r.year, r.month, 1), 'yyyyMMdd') AS INT)
"""

print("=" * 80)
print("STEP 2: Building gold table (FULL load)")
print("=" * 80)
print("Executing query:")
print(lab_test_trends_sql)

spark.sql(lab_test_trends_sql)
print(f"FULL load completed -> {target_table}")

In [0]:
from pyspark.sql import functions as F

print("=" * 80)
print("STEP 3: Validation")
print("=" * 80)

source_tests = spark.table(f"{source_schema}.lab_results").count()
target_df = spark.table(target_table)
target_count = target_df.count()
distinct_keys = target_df.select("test_name", "year", "month").distinct().count()
target_tests = target_df.agg(F.sum("total_tests")).collect()[0][0] or 0

print(f"  Target rows                       : {target_count}")
print(f"  Distinct (test_name, year, month) : {distinct_keys}")
print(f"  Source lab_results rows           : {source_tests}")
print(f"  SUM(total_tests) in target        : {target_tests}")

if distinct_keys != target_count:
    raise ValueError("Grain violation: (test_name, year, month) is not unique in target")
if target_tests != source_tests:
    raise ValueError("Test count mismatch between silver.lab_results and target")
print("Validation passed.")

#display(target_df.orderBy("test_name", "year", "month").limit(10))

In [0]:
record_count = target_df.count()
print(f"Records in {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))